# Чтение данных Inside Airbnb из S3

В этом ноутбуке реализовано чтение подготовленного CSV из S3-совместимого хранилища. При ошибке доступа или сети можно использовать локальную копию того же набора данных.


## Подготовка окружения

В проекте используется отдельное виртуальное окружение `.venv`, а зависимости перечислены в `requirements.txt`. Параметры подключения и ключи находятся в локальном файле `.env`. Файл `.env` и директория данных исключены из Git.


Установка зависимостей в активированном виртуальном окружении:

```bash
python -m pip install -r requirements.txt
```


## Импорты и подключение к S3

Загружаю настройки из `.env` и создаю клиент S3. `S3_BUCKET` — имя bucket, `S3_KEY` — полный путь к CSV внутри него. Переменные окружения, заданные до запуска, имеют приоритет над значениями из `.env`.


In [1]:
import os
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import BotoCoreError, ClientError
from dotenv import load_dotenv

# .env содержит локальные настройки, его нельзя добавлять в Git.
load_dotenv(Path(".env"), override=False)

S3_BUCKET = os.getenv("S3_BUCKET")
S3_KEY = os.getenv("S3_KEY")
S3_ENDPOINT_URL = os.getenv("S3_ENDPOINT_URL") or None
LOCAL_PATH = Path("data/listings.csv")

if not S3_BUCKET or not S3_KEY:
    raise ValueError("Укажите S3_BUCKET и S3_KEY в файле .env.")

# boto3 получает AWS_ACCESS_KEY_ID и AWS_SECRET_ACCESS_KEY из окружения.
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT_URL,
    region_name=os.getenv("AWS_DEFAULT_REGION") or None,
)


## Чтение CSV из S3

Функция `read_s3_csv` получает объект по bucket и key и возвращает его как `pandas.DataFrame`. Столбец `listing_id` читается как строка, чтобы сохранить исходный вид идентификаторов.


In [2]:
def read_s3_csv(bucket, key):
    """Загрузить CSV из S3 и вернуть pandas DataFrame."""
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(response["Body"], dtype={"listing_id": "string"})

## Чтение с локальной копией

Функция `read_data` сначала обращается к S3. При ошибке доступа или сети она читает локальный CSV, если путь к нему указан. Если путь не передан, исходная ошибка передаётся дальше. Ошибки формата CSV не скрываются.


In [3]:
def read_data(bucket, key, local_path=None):
    """Прочитать CSV из S3; при ошибке доступа или сети использовать локальный файл."""
    try:
        return read_s3_csv(bucket, key)
    except (BotoCoreError, ClientError, OSError) as error:
        if local_path is None:
            raise
        print(f"Ошибка {type(error).__name__}: читаем локальный файл {local_path}")
        return pd.read_csv(local_path, dtype={"listing_id": "string"})

## Проверка чтения из S3

Загружаю таблицу и просматриваю первые строки, размер и названия столбцов.


In [ ]:
df_s3 = read_s3_csv(S3_BUCKET, S3_KEY)
display(df_s3.head())
print("Размер таблицы:", df_s3.shape)
print("Колонки:", df_s3.columns.tolist())

## Локальная копия данных

После успешного чтения из S3 сохраняю локальную копию, если файл ещё не существует. Если файл уже есть, необходимо убедиться, что он содержит тот же снимок данных.


In [ ]:
LOCAL_PATH.parent.mkdir(parents=True, exist_ok=True)
if not LOCAL_PATH.exists():
    df_s3.to_csv(LOCAL_PATH, index=False)
    print(f"Локальная копия создана: {LOCAL_PATH}")
else:
    print(f"Файл уже существует: {LOCAL_PATH}. Проверьте, что это тот же снимок данных.")

## Проверка функции `read_data`

При доступном объекте функция возвращает данные из S3.


In [ ]:
df = read_data(S3_BUCKET, S3_KEY, LOCAL_PATH)
display(df.head())
print("Размер:", df.shape)

### Проверка перехода к локальному файлу

Добавляю к ключу суффикс `.missing`, чтобы вызвать ошибку доступа к объекту. При наличии локальной копии функция сообщает о переходе к ней и возвращает таблицу.


In [ ]:
df_local = read_data(S3_BUCKET, S3_KEY + ".missing", LOCAL_PATH)
display(df_local.head())
print("Размер:", df_local.shape)

### Дополнительные проверки ошибок

Вызовы ниже оставлены закомментированными: первый должен передать исходную ошибку S3, второй — вызвать `FileNotFoundError`, если локального файла нет.


In [ ]:
# read_data(S3_BUCKET, S3_KEY + ".missing")
# read_data(S3_BUCKET, S3_KEY + ".missing", "data/absent.csv")